# CMPE 260 Project 2 — Policy Gradients on Atari Pong (REINFORCE, A2C) vs DQN

*Local version for a Windows (or Linux) PC with an NVIDIA RTX 4080. No Google Colab or Drive needed.*

This notebook builds directly on Project 1 (`CMPE260_DQN_Pong_Complete(PER).ipynb`).

| Step | What it does |
|---|---|
| 1 | **REINFORCE** (Williams, 1992): Monte-Carlo policy gradient, the baseline PG method |
| 2 | **Hyper-parameter tuning** of REINFORCE with short, controlled trials |
| 3 | **A2C** (synchronous advantage actor-critic) with its own tuning sweep |
| 4 | **Comparison** of REINFORCE, A2C and the Project 1 DQN runs: convergence speed, value over-estimation and bias, wall-clock time, samples, updates and final score |

**Same setting as Project 1.** The environment is `ALE/Pong-v5` with `frameskip=1` and
`repeat_action_probability=0.0`. The same wrappers are used: 4-frame max-and-skip, FIRE reset,
84×84 grayscale crop and a 4-frame stack. The settings are γ = 0.99, seed 42, the same CNN torso,
the same "solved" rule (mean reward of the last 100 episodes ≥ 19) and the same TensorBoard tag names.
As in Project 1, **"frames" means agent steps**. Each agent step is 4 emulator frames.

**GPU usage.** Both policy-gradient methods run 16 Pong emulators in parallel worker processes.
Each step makes one batched forward pass on the GPU for all 16 environments. The rollout buffers
stay on the GPU. REINFORCE updates in large GPU mini-batches. On an RTX 4080 (Ada, sm_89) the
network runs in bf16 autocast, while losses and probabilities stay in fp32.

The notebook is interrupt-safe. Every run checkpoints to a local folder and resumes automatically
when you re-run the cell. Use **Kernel → Interrupt** to stop cleanly.

## 1. Local setup (one time)

1. **Python 3.10–3.12** with Jupyter (`pip install jupyterlab ipykernel`).
2. **PyTorch with CUDA.** Install the CUDA build of PyTorch first; the default PyPI wheel on Windows is CPU-only:
   ```
   pip install torch --index-url https://download.pytorch.org/whl/cu128
   ```
   An up-to-date NVIDIA driver is enough. You do not need a separate CUDA toolkit.
3. Put this notebook in your project folder and start Jupyter from that folder. Runs are written to
   `./CMPE260_Project2/`.
4. **Project 1 artifacts (optional but recommended).** Download the `CMPE260_Project1` folder from
   Google Drive and place it next to this notebook. It holds the TensorBoard logs and `best.pt` for
   `baseline_epsilon_seed42`, `noisynet_seed42` and `per_noisynet_seed42`.
   * With the folder, the DQN learning curves are plotted and the DQN checkpoints are evaluated for Q over-estimation.
   * Without it, the comparison table uses the Project 1 summary numbers recorded in the Project 1
     notebook outputs (Section 16), and the DQN curve and bias columns show "—".

**Troubleshooting on Windows**
* `OSError: [WinError 1114] ... c10.dll` when importing torch inside Jupyter means an old
  `pyzmq` (≤ 25) is loading a Microsoft C++ runtime that conflicts with PyTorch. Fix it with
  `pip install --upgrade "pyzmq>=26"` (the install cell below already does this), then restart the kernel.
* Make sure the Jupyter kernel is the same Python that has CUDA PyTorch. The first code cells print
  the Python version, the PyTorch version and the GPU name.

**Hardware notes**
* Atari emulation and OpenCV preprocessing run on the **CPU**, with one worker process per
  environment. A modern 8–16-core desktop CPU keeps a 4080 busy; the GPU will not be the bottleneck.
* On Windows the workers are started with `spawn` (Windows has no `fork`). That is why the
  environment code lives in a small module, `pong_envs.py`, which the notebook writes next to itself
  in Section 4. Starting 16 workers takes a few seconds per run.
* Close other GPU-heavy programs while training. One RTX 4080 uses well under 2 GB of VRAM here.
* For parallel training on one PC, open two copies of this notebook with `RUN_REINFORCE=True,
  RUN_A2C=False` in one and the opposite in the other. Both can share the GPU, but each needs
  16 CPU workers, so only do this with 16 or more CPU cores.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
%pip install -q "gymnasium==1.3.0" "ale-py==0.12.1" "opencv-python-headless>=4.12" "tensorboard>=2.15" "pyzmq>=26" pandas matplotlib tabulate psutil

In [ ]:
import collections
import contextlib
import functools
import importlib
import json
import math
import os
import platform
import random
import shutil
import sys
import time
import warnings
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import ale_py
import cv2
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from IPython.display import display
from torch.utils.tensorboard import SummaryWriter

torch.distributions.Distribution.set_default_validate_args(False)  # skip per-step arg checks
gym.register_envs(ale_py)
cv2.setNumThreads(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError(
        "PyTorch cannot see a CUDA GPU. Install the CUDA build of PyTorch "
        "(pip install torch --index-url https://download.pytorch.org/whl/cu128) and restart the kernel."
    )

# GPU speed-ups that do not change the algorithms.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
# bf16 autocast on Ampere/Ada and newer (RTX 30xx/40xx, sm_80+). Older GPUs stay fp32/TF32.
AMP_DTYPE = torch.bfloat16 if GPU_CAPABILITY[0] >= 8 else None
CPU_COUNT = os.cpu_count() or 1
IS_WINDOWS = platform.system() == "Windows"

print("OS        :", platform.platform())
print("Python    :", platform.python_version())
print("Gymnasium :", gym.__version__)
print("ALE       :", ale_py.__version__)
print("OpenCV    :", cv2.__version__)
print("PyTorch   :", torch.__version__, "CUDA", torch.version.cuda)
print("GPU       :", GPU_NAME, "capability", GPU_CAPABILITY,
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
print("AMP dtype :", AMP_DTYPE)
print("CPU cores :", CPU_COUNT, f"(physical {psutil.cpu_count(logical=False)})")
print("RAM       :", f"{psutil.virtual_memory().total / 2**30:.1f} GiB")
if CPU_COUNT < 8:
    print("WARNING: fewer than 8 logical CPUs. Training works but env stepping will be the bottleneck.")

## 2. Local folders

Project 1 results are **read** from `P1_ROOT`, and Project 2 runs are **written** to `P2_ROOT`.
Both default to folders next to this notebook. You can override them with the environment
variables `CMPE260_P1_ROOT` and `CMPE260_P2_ROOT`, or by editing the paths below.

In [ ]:
PROJECT_DIR = Path.cwd()
P1_ROOT = Path(os.environ.get("CMPE260_P1_ROOT", PROJECT_DIR / "CMPE260_Project1"))
P2_ROOT = Path(os.environ.get("CMPE260_P2_ROOT", PROJECT_DIR / "CMPE260_Project2"))
P2_ROOT.mkdir(parents=True, exist_ok=True)

print("Notebook folder    :", PROJECT_DIR)
print("Project 1 artifacts:", P1_ROOT, "(exists)" if P1_ROOT.exists() else
      "(NOT FOUND - comparison will use recorded Project 1 summary numbers)")
if P1_ROOT.exists():
    print("  runs:", sorted(p.name for p in P1_ROOT.iterdir() if p.is_dir()))
print("Project 2 artifacts:", P2_ROOT)
free_gb = shutil.disk_usage(P2_ROOT).free / 2**30
print(f"Free disk at P2_ROOT: {free_gb:.1f} GiB (all runs together need roughly 1-2 GiB)")

## 3. Configuration

One dataclass holds both algorithms' settings, so the controlled differences are explicit.
The values below are the **starting points** before tuning. Section 8 and Section 10 sweep the
most important ones.

| Setting | REINFORCE | A2C | Why |
|---|---|---|---|
| parallel envs | 16 | 16 | same data throughput for both PG methods |
| rollout length | 512 steps/env (8,192 states/update) | 5 steps/env (80 states/update) | MC needs complete returns; TD bootstraps after *n* steps |
| optimizer | Adam, lr 5e-4 | RMSprop(α=0.99, ε=1e-5), lr 7e-4 | standard choices from the A2C paper and its reference implementations |
| entropy bonus | 0.01 | 0.01 | stops the policy from collapsing too early |
| value loss coef | — | 0.5 | |
| grad-norm clip | 1.0 | 0.5 | |
| return processing | per-point discounted return, standardised per batch | n-step / GAE(λ) advantage | variance reduction |

In [ ]:
@dataclass(frozen=True)
class PGConfig:
    algorithm: str                     # "reinforce" or "a2c"
    env_id: str = "ALE/Pong-v5"
    seed: int = 42
    gamma: float = 0.99
    num_envs: int = 16
    rollout_steps: int = 512           # steps per env between updates
    optimizer: str = "adam"            # "adam" or "rmsprop"
    learning_rate: float = 5e-4
    adam_eps: float = 1e-5
    rmsprop_alpha: float = 0.99
    rmsprop_eps: float = 1e-5
    entropy_coef: float = 0.01
    value_coef: float = 0.5            # A2C only
    max_grad_norm: float = 0.5
    gae_lambda: float = 1.0            # A2C only; 1.0 = plain n-step returns
    point_reset: bool = True           # REINFORCE only; reset the return at every Pong point
    normalize_returns: bool = True     # REINFORCE only; standardise returns per batch
    update_chunk: int = 4096           # GPU mini-batch for REINFORCE gradient accumulation
    use_amp: bool = True               # bf16 autocast on Ampere+ GPUs
    max_frames: int = 10_000_000       # agent steps, summed over all envs
    reward_bound: float = 19.0         # identical to Project 1
    min_episodes_to_solve: int = 100   # identical to Project 1
    convergence_threshold: float = 15.0  # Project 1 comparison threshold
    checkpoint_every: int = 250_000
    diagnostic_every: int = 10_000
    best_save_interval: int = 50_000   # rate-limits best.pt writes
    print_every_seconds: float = 30.0


REINFORCE_CONFIG = PGConfig(
    algorithm="reinforce",
    rollout_steps=512,
    optimizer="adam",
    learning_rate=5e-4,
    entropy_coef=0.01,
    value_coef=0.0,
    max_grad_norm=1.0,
    point_reset=True,
    normalize_returns=True,
    max_frames=15_000_000,
)

A2C_CONFIG = PGConfig(
    algorithm="a2c",
    rollout_steps=5,
    optimizer="rmsprop",
    learning_rate=7e-4,
    entropy_coef=0.01,
    value_coef=0.5,
    max_grad_norm=0.5,
    gae_lambda=1.0,
    point_reset=False,
    normalize_returns=False,
    max_frames=10_000_000,
)

ENV_ID, SEED, GAMMA = REINFORCE_CONFIG.env_id, REINFORCE_CONFIG.seed, REINFORCE_CONFIG.gamma
print(json.dumps({"REINFORCE": asdict(REINFORCE_CONFIG), "A2C": asdict(A2C_CONFIG)}, indent=2))

## 4. Atari preprocessing (identical to Project 1)

The wrapper classes are copied unchanged from Project 1. The next cell writes them to `pong_envs.py`
next to the notebook and imports them. A separate module is needed because Windows starts worker
processes with `spawn`, and spawned workers can only rebuild an environment from code they can
import; they cannot use classes defined inside a notebook.

`make_vec_env` runs `num_envs` copies, each in its own process. It uses `SAME_STEP` auto-reset:
when an episode ends, the reset observation comes back on the same step. This keeps rollout
bookkeeping simple and correct.

In [ ]:
PONG_ENVS_SOURCE = r"""
# Auto-generated by CMPE260_Project2_PG_Pong.ipynb. Identical preprocessing to Project 1.
import collections

import ale_py
import cv2
import gymnasium as gym
import numpy as np

gym.register_envs(ale_py)
cv2.setNumThreads(1)  # one OpenCV thread per env worker avoids CPU oversubscription


class MaxAndSkipEnv(gym.Wrapper):
    def __init__(self, env, skip=4):
        super().__init__(env)
        self.skip = int(skip)
        self.frames = collections.deque(maxlen=2)

    def reset(self, **kwargs):
        self.frames.clear()
        obs, info = self.env.reset(**kwargs)
        self.frames.append(obs)
        return obs, info

    def step(self, action):
        total_reward = 0.0
        terminated = truncated = False
        info = {}
        for _ in range(self.skip):
            obs, reward, terminated, truncated, info = self.env.step(action)
            self.frames.append(obs)
            total_reward += float(reward)
            if terminated or truncated:
                break
        return np.maximum.reduce(tuple(self.frames)), total_reward, terminated, truncated, info


class FireResetEnv(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        meanings = env.unwrapped.get_action_meanings()
        if len(meanings) < 3 or meanings[1] != "FIRE":
            raise ValueError(f"Unexpected Pong actions: {meanings}")

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        for action in (1, 2):
            obs, _, terminated, truncated, info = self.env.step(action)
            if terminated or truncated:
                obs, info = self.env.reset()
        return obs, info


class ProcessFrame84(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.observation_space = gym.spaces.Box(low=0, high=255, shape=(1, 84, 84), dtype=np.uint8)

    def observation(self, frame):
        if frame.shape not in ((210, 160, 3), (250, 160, 3)):
            raise ValueError(f"Unexpected Atari frame shape: {frame.shape}")
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        resized = cv2.resize(gray, (84, 110), interpolation=cv2.INTER_AREA)
        return resized[18:102, :][None, ...].astype(np.uint8, copy=False)


class FrameStack4(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        self.frames = collections.deque(maxlen=4)
        self.observation_space = gym.spaces.Box(low=0, high=255, shape=(4, 84, 84), dtype=np.uint8)

    def _stack(self):
        return np.concatenate(tuple(self.frames), axis=0)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        zero = np.zeros_like(obs)
        self.frames.clear()
        self.frames.extend([zero.copy(), zero.copy(), zero.copy(), obs.copy()])
        return self._stack(), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.frames.append(obs.copy())
        return self._stack(), reward, terminated, truncated, info


def make_env(env_id="ALE/Pong-v5"):
    env = gym.make(env_id, frameskip=1, repeat_action_probability=0.0, full_action_space=False)
    env = MaxAndSkipEnv(env, skip=4)
    env = FireResetEnv(env)
    env = ProcessFrame84(env)
    env = FrameStack4(env)
    return env
"""

_module_path = PROJECT_DIR / "pong_envs.py"
if not _module_path.exists() or _module_path.read_text(encoding="utf-8") != PONG_ENVS_SOURCE:
    _module_path.write_text(PONG_ENVS_SOURCE, encoding="utf-8")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
import pong_envs
importlib.reload(pong_envs)
from pong_envs import FireResetEnv, FrameStack4, MaxAndSkipEnv, ProcessFrame84, make_env

VECTOR_BACKEND = "async"  # "sync" runs all envs in this process (slow; debugging only)
MP_CONTEXT = "spawn" if IS_WINDOWS else "fork"


def make_vec_env(num_envs, env_id=ENV_ID, backend=None):
    backend = backend or VECTOR_BACKEND
    fns = [functools.partial(make_env, env_id) for _ in range(num_envs)]  # picklable for spawn
    mode = gym.vector.AutoresetMode.SAME_STEP
    if backend == "async":
        return gym.vector.AsyncVectorEnv(fns, shared_memory=True, context=MP_CONTEXT, autoreset_mode=mode)
    return gym.vector.SyncVectorEnv(fns, autoreset_mode=mode)


print("Wrote", _module_path, "| worker start method:", MP_CONTEXT)

### Preflight test and throughput benchmark

Do not continue if an assertion fails. The benchmark shows the environment-only throughput
ceiling of this runtime, in agent steps per second.

In [ ]:
test_env = make_env()
obs0, _ = test_env.reset(seed=SEED)
obs1, *_ = test_env.step(test_env.action_space.sample())
assert obs0.shape == (4, 84, 84) and obs0.dtype == np.uint8
assert not np.shares_memory(obs0, obs1) and np.any(obs0 != obs1)
print("Single env :", obs0.shape, obs0.dtype, test_env.unwrapped.get_action_meanings())
test_env.close()

bench = make_vec_env(REINFORCE_CONFIG.num_envs)
obs, _ = bench.reset(seed=SEED)
assert obs.shape == (REINFORCE_CONFIG.num_envs, 4, 84, 84)
steps, t0, finished = 0, time.perf_counter(), 0
for _ in range(300):
    obs, r, term, trunc, info = bench.step(bench.action_space.sample())
    steps += REINFORCE_CONFIG.num_envs
    finished += int(np.sum(term | trunc))
bench_fps = steps / (time.perf_counter() - t0)
bench.close()
print(f"Vector env : {REINFORCE_CONFIG.num_envs} envs, {bench_fps:,.0f} agent steps/s (env only, random actions)")

## 5. Networks

All methods use the **same CNN torso** as the Project 1 DQN: conv 32-8/4 → 64-4/2 → 64-3/1 → FC 512.

* **REINFORCE** uses the torso plus a policy head with 6 logits.
* **A2C** uses the torso shared by a policy head and a value head *V(s)*.

Orthogonal initialisation follows the A2C reference implementations: gain √2 in the torso,
0.01 on the policy head (near-uniform initial policy) and 1.0 on the value head.
`DQN` and `NoisyDQN` are copied from Project 1 only so that their checkpoints can be loaded
for evaluation.

In [ ]:
def atari_torso(in_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, 32, kernel_size=8, stride=4), nn.ReLU(),
        nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
        nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
        nn.Flatten(),
    )


class PGNet(nn.Module):
    def __init__(self, input_shape, n_actions, with_value):
        super().__init__()
        self.features = atari_torso(input_shape[0])
        with torch.no_grad():
            feature_size = self.features(torch.zeros(1, *input_shape)).shape[1]
        self.fc = nn.Sequential(nn.Linear(feature_size, 512), nn.ReLU())
        self.policy_head = nn.Linear(512, n_actions)
        self.value_head = nn.Linear(512, 1) if with_value else None
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.orthogonal_(m.weight, gain=math.sqrt(2))
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.policy_head.weight, gain=0.01)
        if self.value_head is not None:
            nn.init.orthogonal_(self.value_head.weight, gain=1.0)

    def forward(self, x):
        h = self.fc(self.features(x.float().div(255.0)))
        logits = self.policy_head(h)
        value = self.value_head(h).squeeze(-1) if self.value_head is not None else None
        return logits, value


def amp_context(enabled):
    if enabled and AMP_DTYPE is not None and device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=AMP_DTYPE)
    return contextlib.nullcontext()


def forward_fp32(net, obs, use_amp):
    # Run the CNN under autocast, but return fp32 logits/values for stable losses.
    with amp_context(use_amp):
        logits, value = net(obs)
    return logits.float(), (None if value is None else value.float())


def build_pg_net(config, obs_shape, n_actions):
    return PGNet(obs_shape, n_actions, with_value=(config.algorithm == "a2c")).to(device)


def build_optimizer(net, config):
    if config.optimizer == "adam":
        try:
            return optim.Adam(net.parameters(), lr=config.learning_rate, eps=config.adam_eps, fused=True)
        except (RuntimeError, TypeError):
            return optim.Adam(net.parameters(), lr=config.learning_rate, eps=config.adam_eps)
    if config.optimizer == "rmsprop":
        return optim.RMSprop(net.parameters(), lr=config.learning_rate,
                             alpha=config.rmsprop_alpha, eps=config.rmsprop_eps)
    raise ValueError(config.optimizer)


def count_params(net):
    return sum(p.numel() for p in net.parameters())


# ---- Project 1 networks (only for loading DQN checkpoints) ----
class DQN(nn.Module):
    def __init__(self, input_shape, n_actions):
        super().__init__()
        self.features = atari_torso(input_shape[0])
        with torch.no_grad():
            feature_size = self.features(torch.zeros(1, *input_shape)).shape[1]
        self.head = nn.Sequential(nn.Linear(feature_size, 512), nn.ReLU(), nn.Linear(512, n_actions))

    def forward(self, x):
        return self.head(self.features(x.float().div(255.0)))


class NoisyLinear(nn.Module):
    def __init__(self, in_features, out_features, sigma0=0.5):
        super().__init__()
        self.in_features, self.out_features = int(in_features), int(out_features)
        self.weight_mu = nn.Parameter(torch.empty(out_features, in_features))
        self.weight_sigma = nn.Parameter(torch.empty(out_features, in_features))
        self.register_buffer("weight_epsilon", torch.zeros(out_features, in_features))
        self.bias_mu = nn.Parameter(torch.empty(out_features))
        self.bias_sigma = nn.Parameter(torch.empty(out_features))
        self.register_buffer("bias_epsilon", torch.zeros(out_features))
        nn.init.zeros_(self.weight_mu); nn.init.zeros_(self.bias_mu)
        nn.init.zeros_(self.weight_sigma); nn.init.zeros_(self.bias_sigma)

    def forward(self, x):
        if self.training:
            w = self.weight_mu + self.weight_sigma * self.weight_epsilon
            b = self.bias_mu + self.bias_sigma * self.bias_epsilon
        else:
            w, b = self.weight_mu, self.bias_mu
        return F.linear(x, w, b)


class NoisyDQN(nn.Module):
    def __init__(self, input_shape, n_actions, sigma0=0.5):
        super().__init__()
        self.features = atari_torso(input_shape[0])
        with torch.no_grad():
            feature_size = self.features(torch.zeros(1, *input_shape)).shape[1]
        self.fc1 = NoisyLinear(feature_size, 512, sigma0)
        self.fc2 = NoisyLinear(512, n_actions, sigma0)

    def forward(self, x):
        x = self.features(x.float().div(255.0))
        return self.fc2(torch.relu(self.fc1(x)))


_probe = PGNet((4, 84, 84), 6, with_value=True)
print("PGNet (A2C) parameters      :", f"{count_params(_probe):,}")
print("PGNet (REINFORCE) parameters:", f"{count_params(PGNet((4, 84, 84), 6, with_value=False)):,}")
print("DQN parameters              :", f"{count_params(DQN((4, 84, 84), 6)):,}")
del _probe

## 6. Shared run bookkeeping

`RunMonitor` matches the Project 1 runner. It logs the same TensorBoard tags
(`episode/reward`, `episode/reward_100`, `system/fps`, `system/wall_minutes`, `train/*`), writes
`latest.pt`, `best.pt` and `results.json` atomically, and applies the Project 1 stopping rule.
It also saves `episodes.npz` with the frame, reward, length and wall-clock time of every episode.
The comparison uses this file because 16 parallel environments can finish episodes on the same
frame, and TensorBoard would merge those events.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def _replace(tmp, path, attempts=20):
    # On Windows, antivirus/indexers can briefly lock a file; retry the atomic rename.
    for i in range(attempts):
        try:
            os.replace(tmp, path)
            return
        except PermissionError:
            if i == attempts - 1:
                raise
            time.sleep(0.25)


def atomic_json(payload, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, default=float), encoding="utf-8")
    _replace(tmp, path)


def atomic_torch_save(payload, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    _replace(tmp, path)


def atomic_npz(path, **arrays):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "wb") as fh:
        np.savez(fh, **arrays)
    _replace(tmp, path)


class EpisodeTracker:
    # Accumulates per-env episode return/length across vectorised steps.
    def __init__(self, n):
        self.returns = np.zeros(n, dtype=np.float64)
        self.lengths = np.zeros(n, dtype=np.int64)

    def step(self, rewards, dones):
        self.returns += rewards
        self.lengths += 1
        finished = []
        for i in np.flatnonzero(dones):
            finished.append((float(self.returns[i]), int(self.lengths[i])))
            self.returns[i] = 0.0
            self.lengths[i] = 0
        return finished


class RunMonitor:
    def __init__(self, run_tag, config, net, optimizer, resume_from=None):
        self.run_tag, self.config, self.net, self.optimizer = run_tag, config, net, optimizer
        self.run_dir = P2_ROOT / run_tag
        self.run_dir.mkdir(parents=True, exist_ok=True)
        self.frame = 0
        self.updates = 0
        self.samples_used = 0
        self.rewards, self.lengths, self.episode_frames, self.episode_wall = [], [], [], []
        self.best_mean = -math.inf
        self.best_saved_mean = -math.inf
        self.milestones = {}
        self.prior_elapsed = 0.0
        if resume_from is not None:
            saved = torch.load(resume_from, map_location=device, weights_only=False)
            if saved.get("algorithm") != config.algorithm:
                raise ValueError("Checkpoint algorithm does not match this run.")
            net.load_state_dict(saved["model"])
            optimizer.load_state_dict(saved["optimizer"])
            self.frame = int(saved["frame"])
            self.updates = int(saved["updates"])
            self.samples_used = int(saved["samples_used"])
            self.rewards = list(saved["rewards"])
            self.lengths = list(saved["lengths"])
            self.episode_frames = list(saved["episode_frames"])
            self.episode_wall = list(saved["episode_wall"])
            self.best_mean = float(saved["best_mean"])
            self.best_saved_mean = float(saved["best_saved_mean"])
            self.milestones = dict(saved.get("milestones", {}))
            self.prior_elapsed = float(saved["elapsed_seconds"])
            print(f"Resumed {run_tag} at frame {self.frame:,} ({len(self.rewards)} episodes).")
        self.writer = SummaryWriter(str(self.run_dir / "tensorboard"), purge_step=self.frame or None)
        self.started = time.perf_counter()
        self.session_start_frame = self.frame
        self.window_started, self.window_frame = self.started, self.frame
        self.last_print = -math.inf
        self.last_checkpoint_frame = self.frame
        self.last_diag_frame = -math.inf
        self.last_best_save_frame = -math.inf
        self.status = "running"
        self.n_params = count_params(net)

    def elapsed(self):
        return self.prior_elapsed + time.perf_counter() - self.started

    @property
    def solved(self):
        return self.status == "solved"

    def on_episode(self, episode_reward, episode_length):
        cfg, f, w = self.config, self.frame, self.writer
        wall_min = self.elapsed() / 60
        self.rewards.append(float(episode_reward))
        self.lengths.append(int(episode_length))
        self.episode_frames.append(int(f))
        self.episode_wall.append(float(wall_min))
        n_eps = len(self.rewards)
        mean_100 = float(np.mean(self.rewards[-100:]))
        w.add_scalar("episode/reward", episode_reward, f)
        w.add_scalar("episode/reward_100", mean_100, f)
        w.add_scalar("episode/length", episode_length, f)
        w.add_scalar("system/wall_minutes", wall_min, f)

        for name, hit in ((f"+{cfg.convergence_threshold:g}", mean_100 >= cfg.convergence_threshold),
                          ("solved", n_eps >= cfg.min_episodes_to_solve and mean_100 >= cfg.reward_bound)):
            if hit and name not in self.milestones:
                self.milestones[name] = {"frame": f, "episodes": n_eps, "wall_minutes": wall_min}
                print(f"*** {self.run_tag}: reached {name} at frame {f:,}, episode {n_eps}, {wall_min:.1f} min")

        if n_eps >= 10 and mean_100 > self.best_mean:
            self.best_mean = mean_100
        if (n_eps >= 10 and mean_100 > self.best_saved_mean
                and (f - self.last_best_save_frame >= cfg.best_save_interval or mean_100 >= cfg.reward_bound)):
            self.best_saved_mean = mean_100
            self.last_best_save_frame = f
            atomic_torch_save(self.checkpoint_payload(), self.run_dir / "best.pt")

        now = time.perf_counter()
        if now - self.last_print >= cfg.print_every_seconds:
            fps = (f - self.window_frame) / max(now - self.window_started, 1e-9)
            self.window_frame, self.window_started = f, now
            w.add_scalar("system/fps", fps, f)
            print(f"{self.run_tag} frame={f:>11,} episode={n_eps:>6} reward={episode_reward:6.1f} "
                  f"mean100={mean_100:7.3f} updates={self.updates:>8,} fps={fps:7.1f} "
                  f"wall={wall_min:6.1f}m")
            self.last_print = now

        if n_eps >= cfg.min_episodes_to_solve and mean_100 >= cfg.reward_bound:
            self.status = "solved"

    def on_update(self, samples, diagnostics):
        self.updates += 1
        self.samples_used += int(samples)
        if self.frame - self.last_diag_frame >= self.config.diagnostic_every:
            for key, value in diagnostics.items():
                self.writer.add_scalar(f"train/{key}", float(value), self.frame)
            self.last_diag_frame = self.frame
        if self.frame - self.last_checkpoint_frame >= self.config.checkpoint_every:
            self.save_latest()
            self.last_checkpoint_frame = self.frame

    def checkpoint_payload(self):
        return {
            "model": self.net.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "algorithm": self.config.algorithm,
            "frame": self.frame,
            "updates": self.updates,
            "samples_used": self.samples_used,
            "rewards": self.rewards,
            "lengths": self.lengths,
            "episode_frames": self.episode_frames,
            "episode_wall": self.episode_wall,
            "best_mean": self.best_mean,
            "best_saved_mean": self.best_saved_mean,
            "milestones": self.milestones,
            "elapsed_seconds": self.elapsed(),
            "config": asdict(self.config),
            "gpu": GPU_NAME,
        }

    def result_payload(self):
        elapsed = self.elapsed()
        return {
            "run_tag": self.run_tag,
            "algorithm": self.config.algorithm,
            "status": self.status,
            "frames": self.frame,
            "env_frames": 4 * self.frame,
            "episodes": len(self.rewards),
            "updates": self.updates,
            "samples_used": self.samples_used,
            "mean_reward_100": float(np.mean(self.rewards[-100:])) if self.rewards else None,
            "best_mean_reward_100": None if self.best_mean == -math.inf else self.best_mean,
            "milestones": self.milestones,
            "wall_clock_seconds": elapsed,
            "wall_clock_minutes": elapsed / 60,
            "avg_fps": self.frame / max(elapsed, 1e-9),
            "gpu": GPU_NAME,
            "cpu_count": CPU_COUNT,
            "num_params": self.n_params,
            "config": asdict(self.config),
        }

    def save_latest(self):
        atomic_torch_save(self.checkpoint_payload(), self.run_dir / "latest.pt")
        atomic_npz(self.run_dir / "episodes.npz",
                   frames=np.asarray(self.episode_frames, dtype=np.int64),
                   rewards=np.asarray(self.rewards, dtype=np.float32),
                   lengths=np.asarray(self.lengths, dtype=np.int64),
                   wall_minutes=np.asarray(self.episode_wall, dtype=np.float64))
        atomic_json(self.result_payload(), self.run_dir / "results.json")

    def finish(self, status):
        self.status = status
        self.save_latest()
        self.writer.flush()
        self.writer.close()
        print(json.dumps({k: v for k, v in self.result_payload().items() if k != "config"}, indent=2))

## 7. REINFORCE (Monte-Carlo policy gradient)

$$\nabla_\theta J(\theta)=\mathbb{E}_{\pi_\theta}\Big[\sum_t \nabla_\theta\log\pi_\theta(a_t\mid s_t)\,G_t\Big],\qquad G_t=\sum_{k\ge 0}\gamma^k r_{t+k}.$$

REINFORCE uses the actual sampled return $G_t$, with no critic and no bootstrapping. The gradient
is therefore **unbiased** but has **high variance**. The implementation uses the standard
variance-reduction steps for Pong:

1. **Per-point returns** (`point_reset=True`). Every Pong point ends with reward ±1, so the
   discounted sum restarts after each point. An action only gets credit for the rally it was part
   of. This is the approach from Karpathy's "Pong from Pixels". It does not change which policy is
   optimal, because points are independent.
2. **Return standardisation** within each batch. This works as a constant baseline and fixes the
   gradient scale.
3. **Entropy bonus** and **gradient-norm clipping**.

**Staying strictly on-policy.** Each update uses 16 envs × `rollout_steps` steps. Only steps
whose return is *complete* inside the rollout are used, meaning their rally ended before the
rollout ended. Each env's last, unfinished rally is dropped (about 10–15 % of steps at T = 512),
so no off-policy data is ever used. The batch can hold up to 8,192 states. The GPU processes it
in chunks of `update_chunk` states with gradient accumulation, so the result is exactly one
gradient step on the full batch.

In [ ]:
def discounted_returns_np(rewards, dones, gamma, point_reset):
    # rewards, dones: [T, N]. Returns G_t and a mask of steps whose return is complete in the rollout.
    T, N = rewards.shape
    returns = np.zeros((T, N), dtype=np.float32)
    mask = np.zeros((T, N), dtype=bool)
    running = np.zeros(N, dtype=np.float32)
    closed = np.zeros(N, dtype=bool)
    for t in range(T - 1, -1, -1):
        end = dones[t] | (rewards[t] != 0) if point_reset else dones[t].copy()
        running = rewards[t] + gamma * running * (~end)
        returns[t] = running
        closed |= end
        mask[t] = closed
    return returns, mask


def reinforce_update(net, optimizer, config, obs_buf, act_buf, rew_np, done_np, use_amp):
    returns_np, mask_np = discounted_returns_np(rew_np, done_np, config.gamma, config.point_reset)
    mask = torch.from_numpy(mask_np.reshape(-1)).to(device)
    idx = mask.nonzero(as_tuple=True)[0]
    n = int(idx.numel())
    if n < 2:
        return 0, None
    obs_flat = obs_buf.flatten(0, 1)
    act_flat = act_buf.flatten()
    G = torch.from_numpy(returns_np.reshape(-1)).to(device)[idx]
    raw_mean, raw_std = G.mean(), G.std(correction=0)
    if config.normalize_returns:
        G = (G - raw_mean) / (raw_std + 1e-8)

    optimizer.zero_grad(set_to_none=True)
    pg_total = torch.zeros((), device=device)
    ent_total = torch.zeros((), device=device)
    chunk = min(config.update_chunk, obs_flat.shape[0])
    for start in range(0, n, chunk):
        sel = idx[start:start + chunk]
        g = G[start:start + chunk]
        w = torch.ones_like(g)
        pad = chunk - sel.numel()
        if pad:
            # Pad to a FIXED batch shape: with cudnn.benchmark, every new shape triggers a slow
            # kernel re-tune (measured ~25x slower). Padding rows get weight 0 -> exact gradient.
            sel = torch.cat([sel, sel[:1].expand(pad)])
            g = torch.cat([g, g.new_zeros(pad)])
            w = torch.cat([w, w.new_zeros(pad)])
        logits, _ = forward_fp32(net, obs_flat[sel], use_amp)
        logp_all = F.log_softmax(logits, dim=-1)
        logp = logp_all.gather(1, act_flat[sel].unsqueeze(1)).squeeze(1)
        entropy = -(logp_all.exp() * logp_all).sum(-1)
        pg_sum = -(logp * g * w).sum()
        ent_sum = (entropy * w).sum()
        ((pg_sum - config.entropy_coef * ent_sum) / n).backward()
        pg_total += pg_sum.detach()
        ent_total += ent_sum.detach()
    grad_norm = nn.utils.clip_grad_norm_(net.parameters(), config.max_grad_norm)
    optimizer.step()
    diagnostics = {
        "policy_loss": pg_total / n,
        "entropy": ent_total / n,
        "grad_norm": grad_norm,
        "return_mean": raw_mean,
        "return_std": raw_std,
        "batch_size": float(n),
        "used_fraction": n / mask_np.size,
    }
    return n, diagnostics

## 8. A2C (synchronous advantage actor-critic)

The actor uses the advantage $A_t=\hat R_t - V(s_t)$ instead of the full Monte-Carlo return. The
critic $V_\phi$ is trained on the bootstrapped *n*-step target, or on GAE(λ) when `gae_lambda < 1`:

$$\hat R_t=\sum_{k=0}^{n-1}\gamma^k r_{t+k}+\gamma^n V_\phi(s_{t+n}),\qquad
\mathcal L=-\mathbb E[\log\pi(a_t|s_t)A_t]+c_v\,\mathbb E[(\hat R_t-V(s_t))^2]-c_e\,\mathcal H[\pi].$$

Bootstrapping lowers variance compared with REINFORCE, at the cost of some **bias** from the
critic's errors. That bias is measured in Section 12. All 16 environments step in lock-step, and
one update is made every `rollout_steps` steps. Episodes that end by time-limit truncation (rare
in Pong) get their final state's value added as a bootstrap, rather than being treated as
terminal.

In [ ]:
def compute_gae(rewards, dones, values, last_value, gamma, lam):
    # All tensors [T, N] (last_value [N]). dones[t]=1 means the episode ended at step t.
    T = rewards.shape[0]
    advantages = torch.zeros_like(rewards)
    gae = torch.zeros_like(last_value)
    for t in range(T - 1, -1, -1):
        next_value = last_value if t == T - 1 else values[t + 1]
        nonterminal = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_value * nonterminal - values[t]
        gae = delta + gamma * lam * nonterminal * gae
        advantages[t] = gae
    return advantages, advantages + values


def a2c_update(net, optimizer, config, obs_buf, act_buf, rew_np, done_np, val_buf, last_obs, use_amp):
    T, N = act_buf.shape
    rewards = torch.from_numpy(rew_np).to(device)
    dones = torch.from_numpy(done_np.astype(np.float32)).to(device)
    with torch.no_grad():
        _, last_value = forward_fp32(net, last_obs, use_amp)
        advantages, returns = compute_gae(rewards, dones, val_buf, last_value, config.gamma, config.gae_lambda)
    logits, values = forward_fp32(net, obs_buf.flatten(0, 1), use_amp)
    logp_all = F.log_softmax(logits, dim=-1)
    logp = logp_all.gather(1, act_buf.flatten().unsqueeze(1)).squeeze(1)
    entropy = -(logp_all.exp() * logp_all).sum(-1).mean()
    adv = advantages.flatten()
    ret = returns.flatten()
    policy_loss = -(logp * adv).mean()
    value_loss = F.mse_loss(values, ret)
    loss = policy_loss + config.value_coef * value_loss - config.entropy_coef * entropy

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    grad_norm = nn.utils.clip_grad_norm_(net.parameters(), config.max_grad_norm)
    optimizer.step()

    with torch.no_grad():
        explained_var = 1.0 - (ret - values).var() / ret.var().clamp_min(1e-8)
        diagnostics = {
            "loss": loss.detach(),
            "policy_loss": policy_loss.detach(),
            "value_loss": value_loss.detach(),
            "entropy": entropy.detach(),
            "grad_norm": grad_norm,
            "value_mean": values.mean(),        # critic estimate  (cf. DQN train/q_mean)
            "target_mean": ret.mean(),          # bootstrapped target (cf. DQN train/target_mean)
            "advantage_std": adv.std(),
            "explained_variance": explained_var,
        }
    return T * N, diagnostics

## 9. Shared vectorised trainer

The rollout and bookkeeping loop is the same for both algorithms. Only the update rule
(`reinforce_update` or `a2c_update`) and the rollout length differ.

In [ ]:
def train_pg(run_tag, config, resume_from=None):
    if config.algorithm not in {"reinforce", "a2c"}:
        raise ValueError(config.algorithm)
    set_seed(config.seed)
    envs = make_vec_env(config.num_envs, config.env_id)
    obs_shape = envs.single_observation_space.shape
    n_actions = int(envs.single_action_space.n)
    net = build_pg_net(config, obs_shape, n_actions)
    optimizer = build_optimizer(net, config)
    mon = RunMonitor(run_tag, config, net, optimizer, resume_from=resume_from)
    tracker = EpisodeTracker(config.num_envs)
    use_amp = bool(config.use_amp and AMP_DTYPE is not None)
    is_a2c = config.algorithm == "a2c"
    T, N = config.rollout_steps, config.num_envs

    # Rollout storage lives on the GPU (uint8 frames -> ~28 KB per state).
    obs_buf = torch.empty((T, N, *obs_shape), dtype=torch.uint8, device=device)
    act_buf = torch.empty((T, N), dtype=torch.long, device=device)
    val_buf = torch.empty((T, N), dtype=torch.float32, device=device) if is_a2c else None
    rew_np = np.zeros((T, N), dtype=np.float32)
    done_np = np.zeros((T, N), dtype=bool)

    obs, _ = envs.reset(seed=config.seed + mon.frame)
    obs_t = torch.from_numpy(obs).to(device)
    status = "error"
    try:
        while mon.frame < config.max_frames and not mon.solved:
            for t in range(T):
                obs_buf[t].copy_(obs_t)
                with torch.no_grad():
                    logits, value = forward_fp32(net, obs_t, use_amp)
                    actions = torch.distributions.Categorical(logits=logits).sample()
                act_buf[t] = actions
                if is_a2c:
                    val_buf[t] = value
                next_obs, rewards, terminated, truncated, infos = envs.step(actions.cpu().numpy())
                rewards = np.asarray(rewards, dtype=np.float32)
                dones = np.logical_or(terminated, truncated)
                stored = rewards
                if is_a2c:
                    trunc_only = np.logical_and(truncated, np.logical_not(terminated))
                    if trunc_only.any():  # bootstrap time-limit truncations from V(final_obs)
                        ids = np.flatnonzero(trunc_only)
                        final = np.zeros((N, *obs_shape), dtype=np.uint8)  # fixed shape for cuDNN
                        for i in ids:
                            final[i] = infos["final_obs"][i]
                        with torch.no_grad():
                            _, v_final = forward_fp32(net, torch.from_numpy(final).to(device), use_amp)
                        stored = rewards.copy()
                        stored[ids] += config.gamma * v_final.cpu().numpy()[ids]
                rew_np[t] = stored
                done_np[t] = dones
                mon.frame += N
                for ep_reward, ep_len in tracker.step(rewards, dones):
                    mon.on_episode(ep_reward, ep_len)
                obs_t = torch.from_numpy(next_obs).to(device, non_blocking=True)
                if mon.solved:
                    break
            if mon.solved:
                break
            if is_a2c:
                samples, diag = a2c_update(net, optimizer, config, obs_buf, act_buf,
                                           rew_np, done_np, val_buf, obs_t, use_amp)
            else:
                samples, diag = reinforce_update(net, optimizer, config, obs_buf, act_buf,
                                                 rew_np, done_np, use_amp)
            if diag is not None:
                for v in diag.values():
                    if torch.is_tensor(v) and not torch.isfinite(v).all():
                        raise RuntimeError(f"Non-finite diagnostic at frame {mon.frame:,}")
                mon.on_update(samples, diag)
        status = "solved" if mon.solved else "max_frames"
    except KeyboardInterrupt:
        status = "interrupted"
        print("Interrupted; checkpoint saved. Re-run the cell to resume.")
    finally:
        mon.finish(status)
        envs.close()
    return mon.result_payload()


FORCE_RESTART = False  # True deletes the run folder and trains from scratch


def run_or_resume(run_tag, config):
    run_dir = P2_ROOT / run_tag
    if FORCE_RESTART and run_dir.exists():
        print(f"FORCE_RESTART: deleting {run_dir}")
        shutil.rmtree(run_dir)
    results_path, latest = run_dir / "results.json", run_dir / "latest.pt"
    if results_path.exists():
        prev = json.loads(results_path.read_text(encoding="utf-8"))
        if prev.get("status") == "solved" or (
            prev.get("status") == "max_frames" and prev.get("frames", 0) >= config.max_frames
        ):
            print(f"Using completed run: {run_tag} ({prev['status']}, {prev['frames']:,} frames)")
            return prev
    return train_pg(run_tag, config, resume_from=latest if latest.exists() else None)

## 10. Correctness checks (run these before any long training)

* The return and GAE calculations are checked against hand-computed values.
* A gradient-direction check confirms that the policy-gradient loss raises the log-probability of
  positive-advantage actions and lowers it for negative-advantage actions.
* A critic check confirms that the value loss can fit fixed targets.
* Two smoke runs of about 30k frames exercise the full vectorised loop, checkpointing and logging.

In [ ]:
def correctness_checks():
    g = 0.99
    # --- REINFORCE per-point returns ---
    r = np.array([[0], [0], [1], [0], [-1], [0]], dtype=np.float32)
    d = np.zeros_like(r, dtype=bool)
    G, m = discounted_returns_np(r, d, g, point_reset=True)
    assert np.allclose(G[:5, 0], [g * g, g, 1.0, -g, -1.0]), G[:, 0]
    assert m[:, 0].tolist() == [True] * 5 + [False]
    _, m2 = discounted_returns_np(r, d, g, point_reset=False)
    assert not m2.any(), "without a terminal no full-episode return is complete"
    d2 = d.copy(); d2[5] = True
    G3, m3 = discounted_returns_np(r, d2, g, point_reset=False)
    assert m3.all() and np.isclose(G3[0, 0], g ** 2 - g ** 4)

    # --- GAE(lambda=1) equals the n-step bootstrapped return ---
    torch.manual_seed(0)
    T, N = 5, 3
    rew, val, last = torch.randn(T, N), torch.randn(T, N), torch.randn(N)
    dn = torch.zeros(T, N); dn[2, 1] = 1.0
    _, ret = compute_gae(rew, dn, val, last, g, 1.0)
    for n_ in range(N):
        for t in range(T):
            acc, disc, ended = 0.0, 1.0, False
            for k in range(t, T):
                acc += disc * rew[k, n_].item()
                if dn[k, n_] > 0:
                    ended = True
                    break
                disc *= g
            if not ended:
                acc += disc * last[n_].item()
            assert abs(acc - ret[t, n_].item()) < 1e-4

    # --- policy-gradient direction ---
    set_seed(SEED)
    net = PGNet((4, 84, 84), 6, with_value=True).to(device)
    opt = optim.Adam(net.parameters(), lr=1e-4)
    obs = torch.randint(0, 256, (64, 4, 84, 84), dtype=torch.uint8, device=device)
    acts = torch.randint(0, 6, (64,), device=device)
    adv = torch.where(acts < 3, 1.0, -1.0)  # actions 0-2 good, 3-5 bad

    def logp():
        logits, _ = forward_fp32(net, obs, False)
        return F.log_softmax(logits, -1).gather(1, acts[:, None]).squeeze(1)

    before = logp().detach()
    for _ in range(5):  # a few small steps: the PG objective is unbounded, so don't over-train
        loss = -(logp() * adv).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    delta = logp().detach() - before
    assert delta[adv > 0].mean() > 0 > delta[adv < 0].mean(), "PG loss moves log-probs the wrong way"

    # --- critic can fit fixed targets ---
    targets = torch.randn(64, device=device)
    opt = optim.Adam(net.parameters(), lr=1e-3)
    losses = []
    for _ in range(60):
        _, v = forward_fp32(net, obs, False)
        vl = F.mse_loss(v, targets)
        opt.zero_grad(); vl.backward(); opt.step()
        losses.append(vl.item())
    assert losses[-1] < 0.5 * losses[0], losses
    print("Returns, GAE, policy-gradient direction and critic-fit checks passed.")


correctness_checks()

In [ ]:
RUN_SMOKE_TESTS = True

if RUN_SMOKE_TESTS:
    for tag, cfg in [
        ("smoke_reinforce", replace(REINFORCE_CONFIG, rollout_steps=256, max_frames=16 * 256 * 8,
                                    checkpoint_every=16 * 256 * 4, diagnostic_every=1, print_every_seconds=10)),
        ("smoke_a2c", replace(A2C_CONFIG, max_frames=30_000, checkpoint_every=10_000,
                              diagnostic_every=1, print_every_seconds=10)),
    ]:
        shutil.rmtree(P2_ROOT / tag, ignore_errors=True)
        res = train_pg(tag, cfg)
        assert res["frames"] >= cfg.max_frames and res["updates"] > 0, res
        assert (P2_ROOT / tag / "latest.pt").exists() and (P2_ROOT / tag / "episodes.npz").exists()
        print(f"{tag}: OK  ({res['avg_fps']:.0f} agent steps/s, {res['updates']} updates)\n")

## 11. TensorBoard

Project 2 runs (`reinforce_seed42`, `a2c_seed42`, sweep trials) appear next to the Project 1 DQN
runs. They share the same `episode/reward_100` tag, so the curves overlay directly. Useful A2C tags
are `train/value_mean` against `train/target_mean`, which mirror DQN's `train/q_mean` against
`train/target_mean`, and `train/explained_variance`. For REINFORCE, look at `train/return_std`,
`train/grad_norm` and `train/entropy`.

In [ ]:
_specs = [f"p2:{P2_ROOT.as_posix()}"]
for name, tag in [("dqn_baseline", "baseline_epsilon_seed42"), ("dqn_noisynet", "noisynet_seed42"),
                  ("dqn_noisynet_per", "per_noisynet_seed42"), ("dqn_boltzmann", "boltzmann_softmax_seed42")]:
    if (P1_ROOT / tag / "tensorboard").exists():
        _specs.append(f"{name}:{(P1_ROOT / tag / 'tensorboard').as_posix()}")
TB_SPEC = ",".join(_specs)
print(TB_SPEC)
%load_ext tensorboard
%tensorboard --logdir_spec "$TB_SPEC" --port 6006
# The dashboard also opens in a browser at http://localhost:6006

## 12. Hyper-parameter tuning

Each candidate is a **short, controlled trial**. Seed, environment, network and frame budget are
the same for all trials, and only the listed values change. Trials are ranked by their final
`reward_100` (the mean of the last 100 episodes). Ties are broken by the normalised area under the
learning curve, which rewards fast early learning.

* **REINFORCE grid:** learning rate × batch size (rollout length), the two settings that matter
  most for a high-variance Monte-Carlo estimator. Each trial runs 3M frames.
* **A2C grid:** learning rate, optimizer, rollout length and GAE λ. Each trial runs 2M frames.

`REUSE_SWEEP_WINNER = True` continues the winning trial up to the full frame budget instead of
restarting it, which saves 2–3M frames. Its learning curve continues from frame 0 without a gap.
If a sweep is turned off, the defaults from Section 3 are used, or the winner saved from an earlier
session.

Rough cost on an RTX 4080 desktop: throughput is set by the CPU, at about 2–5k agent steps/s
depending on core count (the preflight benchmark in Section 4 shows your ceiling). That is about
10–25 min per REINFORCE trial and 8–20 min per A2C trial. The full runs (15M and 10M frames) take
roughly 1–2 h each, less if a run reaches the solved criterion early.

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator


def tensorboard_curve(event_dir, scalar="episode/reward_100"):
    event_dir = Path(event_dir)
    if not event_dir.exists():
        return np.array([]), np.array([])
    acc = EventAccumulator(str(event_dir), size_guidance={"scalars": 0})
    acc.Reload()
    if scalar not in acc.Tags().get("scalars", []):
        return np.array([]), np.array([])
    by_step = {e.step: e.value for e in acc.Scalars(scalar)}
    steps = np.asarray(sorted(by_step), dtype=np.int64)
    return steps, np.asarray([by_step[s] for s in steps], dtype=np.float64)


def p2_episode_log(run_tag):
    path = P2_ROOT / run_tag / "episodes.npz"
    if not path.exists():
        return None
    d = np.load(path)
    df = pd.DataFrame({"frame": d["frames"], "reward": d["rewards"], "wall_minutes": d["wall_minutes"]})
    df["reward_100"] = df.reward.rolling(100, min_periods=1).mean()
    return df


def normalized_auc(frames, reward_100, horizon):
    # Map Pong reward [-21, 21] -> [0, 1] and average over [0, horizon] (Project 1 definition).
    frames, reward_100 = np.asarray(frames), np.asarray(reward_100)
    if len(frames) == 0:
        return float("nan")
    grid = np.linspace(0, horizon, 2001)
    y = np.interp(grid, frames, reward_100, left=reward_100[0], right=reward_100[-1])
    y = np.clip((y + 21.0) / 42.0, 0.0, 1.0)
    return float(np.trapezoid(y, grid) / max(horizon, 1))


def run_sweep(prefix, base_config, grid, trial_frames):
    rows = []
    for name, overrides in grid.items():
        cfg = replace(base_config, **overrides, max_frames=trial_frames)
        tag = f"sweep_{prefix}_{name}"
        print(f"\n===== {tag}: {overrides} =====")
        res = run_or_resume(tag, cfg)
        log = p2_episode_log(tag)
        auc = normalized_auc(log.frame, log.reward_100, trial_frames) if log is not None else float("nan")
        rows.append({"trial": name, "run_tag": tag, **overrides,
                     "status": res["status"], "frames": res["frames"],
                     "final_reward_100": res["mean_reward_100"],
                     "best_reward_100": res["best_mean_reward_100"],
                     "normalized_auc": auc,
                     "minutes": round(res["wall_clock_minutes"], 1),
                     "fps": round(res["avg_fps"], 1)})
    df = pd.DataFrame(rows).sort_values(["final_reward_100", "normalized_auc"], ascending=False)
    df = df.reset_index(drop=True)
    df.to_csv(P2_ROOT / f"sweep_{prefix}.csv", index=False)
    display(df)
    return df


def choose_config(prefix, base_config, grid, sweep_df, full_frames):
    choice_path = P2_ROOT / f"chosen_{prefix}.json"
    if sweep_df is not None and len(sweep_df):
        winner = sweep_df.iloc[0]["trial"]
        atomic_json({"trial": winner, "overrides": grid[winner]}, choice_path)
    elif choice_path.exists():
        winner = json.loads(choice_path.read_text(encoding="utf-8"))["trial"]
    else:
        winner = None
    overrides = grid.get(winner, {}) if winner else {}
    cfg = replace(base_config, **overrides, max_frames=full_frames)
    print(f"{prefix}: chosen trial = {winner or 'defaults'} -> {overrides}")
    return cfg, winner


def promote_trial(prefix, winner, full_tag):
    # Continue the winning trial as the full run (copies checkpoints + TensorBoard + episode log).
    if winner is None:
        return
    src, dst = P2_ROOT / f"sweep_{prefix}_{winner}", P2_ROOT / full_tag
    if dst.exists() or not (src / "latest.pt").exists():
        return
    shutil.copytree(src, dst)
    print(f"Promoted {src.name} -> {full_tag}; training continues from frame "
          f"{json.loads((dst / 'results.json').read_text(encoding='utf-8'))['frames']:,}")

## 13. REINFORCE: tune, then train

In [ ]:
RUN_REINFORCE = True
RUN_REINFORCE_SWEEP = True
REUSE_SWEEP_WINNER = True
REINFORCE_TRIAL_FRAMES = 3_000_000
REINFORCE_FULL_FRAMES = 15_000_000
REINFORCE_TAG = "reinforce_seed42"

REINFORCE_GRID = {
    "lr2.5e-4_T512": dict(learning_rate=2.5e-4, rollout_steps=512),
    "lr5e-4_T512":   dict(learning_rate=5e-4,   rollout_steps=512),
    "lr1e-3_T512":   dict(learning_rate=1e-3,   rollout_steps=512),
    "lr5e-4_T256":   dict(learning_rate=5e-4,   rollout_steps=256),
}

reinforce_sweep = None
if RUN_REINFORCE and RUN_REINFORCE_SWEEP:
    reinforce_sweep = run_sweep("reinforce", REINFORCE_CONFIG, REINFORCE_GRID, REINFORCE_TRIAL_FRAMES)

BEST_REINFORCE_CONFIG, REINFORCE_WINNER = choose_config(
    "reinforce", REINFORCE_CONFIG, REINFORCE_GRID, reinforce_sweep, REINFORCE_FULL_FRAMES)

In [ ]:
if RUN_REINFORCE:
    if REUSE_SWEEP_WINNER:
        promote_trial("reinforce", REINFORCE_WINNER, REINFORCE_TAG)
    reinforce_results = run_or_resume(REINFORCE_TAG, BEST_REINFORCE_CONFIG)
else:
    print("REINFORCE training disabled in this runtime.")

## 14. A2C: tune, then train

In [ ]:
RUN_A2C = True
RUN_A2C_SWEEP = True
A2C_TRIAL_FRAMES = 2_000_000
A2C_FULL_FRAMES = 10_000_000
A2C_TAG = "a2c_seed42"

A2C_GRID = {
    "rms7e-4_n5":           dict(),                                   # A2C paper / reference defaults
    "rms7e-4_n5_gae0.95":   dict(gae_lambda=0.95),
    "rms3.5e-4_n5":         dict(learning_rate=3.5e-4),
    "adam2.5e-4_n20_gae0.95": dict(optimizer="adam", learning_rate=2.5e-4, rollout_steps=20, gae_lambda=0.95),
}

a2c_sweep = None
if RUN_A2C and RUN_A2C_SWEEP:
    a2c_sweep = run_sweep("a2c", A2C_CONFIG, A2C_GRID, A2C_TRIAL_FRAMES)

BEST_A2C_CONFIG, A2C_WINNER = choose_config("a2c", A2C_CONFIG, A2C_GRID, a2c_sweep, A2C_FULL_FRAMES)

In [ ]:
if RUN_A2C:
    if REUSE_SWEEP_WINNER:
        promote_trial("a2c", A2C_WINNER, A2C_TAG)
    a2c_results = run_or_resume(A2C_TAG, BEST_A2C_CONFIG)
else:
    print("A2C training disabled in this runtime.")

## 15. Evaluation and value-estimate bias / over-estimation

Each method's `best.pt` plays 30 seeded evaluation episodes:

* **DQN:** greedy, and ε = 0.02 as in Project 1. NoisyNet uses its mean weights.
* **REINFORCE / A2C:** greedy (argmax of π), and stochastic (sampling from π, the policy that was trained).

**Bias measurement.** At every step the method's own value estimate is recorded:
$Q(s,a)$ for DQN and $V(s)$ for A2C. After the episode it is compared with the **actual discounted
return** $G_t=\sum_k\gamma^k r_{t+k}$ from that step onward, with γ = 0.99.

$$\text{bias}=\mathbb E[\hat Q(s_t,a_t)-G_t]\quad(\text{DQN}),\qquad \text{bias}=\mathbb E[\hat V(s_t)-G_t]\quad(\text{A2C}).$$

A positive value means **over-estimation**. DQN is expected to over-estimate because of the max
operator in its target. A2C's critic is bootstrapped, but its targets use the sampled action rather
than a max, so its bias is usually smaller. The A2C critic estimates the value of the *stochastic*
policy, so its bias is measured in stochastic mode. REINFORCE has no learned value estimate; its
gradient uses the true sampled return, so it is unbiased by construction, and its cost is the
variance reported in `train/return_std`.

Note: with `repeat_action_probability=0` Pong is deterministic, so greedy evaluation can repeat the
same game in every episode (std = 0, as seen with NoisyNet in Project 1). The ε and stochastic
modes show the spread.

In [ ]:
EVAL_EPISODES = 30
RERUN_EVAL = False


def discounted_episode_returns(rewards, gamma=GAMMA):
    out, running = np.zeros(len(rewards), dtype=np.float64), 0.0
    for t in range(len(rewards) - 1, -1, -1):
        running = rewards[t] + gamma * running
        out[t] = running
    return out


@torch.no_grad()
def play_and_measure(policy_fn, episodes=EVAL_EPISODES, seed_offset=20_000):
    env = make_env()
    scores, lengths, estimates, mc_returns = [], [], [], []
    for ep in range(episodes):
        state, _ = env.reset(seed=SEED + seed_offset + ep)
        ep_rewards, ep_est, done = [], [], False
        while not done:
            action, estimate = policy_fn(state, env)
            state, r, terminated, truncated, _ = env.step(action)
            ep_rewards.append(float(r))
            ep_est.append(estimate)
            done = bool(terminated or truncated)
        scores.append(sum(ep_rewards))
        lengths.append(len(ep_rewards))
        estimates.extend(ep_est)
        mc_returns.extend(discounted_episode_returns(ep_rewards))
    env.close()
    est, mc = np.asarray(estimates, dtype=np.float64), np.asarray(mc_returns, dtype=np.float64)
    result = {"episodes": episodes, "mean": float(np.mean(scores)), "std": float(np.std(scores)),
              "min": float(np.min(scores)), "max": float(np.max(scores)), "scores": scores,
              "mean_length": float(np.mean(lengths))}
    finite = np.isfinite(est)
    if finite.any():
        err = est[finite] - mc[finite]
        rng = np.random.default_rng(0)
        keep = rng.choice(finite.sum(), size=min(4000, finite.sum()), replace=False)
        result.update({
            "estimate_mean": float(est[finite].mean()),
            "mc_return_mean": float(mc[finite].mean()),
            "bias_mean": float(err.mean()),
            "abs_error_mean": float(np.abs(err).mean()),
            "corr": float(np.corrcoef(est[finite], mc[finite])[0, 1]) if finite.sum() > 2 else float("nan"),
            "overestimated_fraction": float((err > 0).mean()),
            "sample_estimates": est[finite][keep].tolist(),
            "sample_returns": mc[finite][keep].tolist(),
        })
    return result


def load_dqn(run_tag):
    path = P1_ROOT / run_tag / "best.pt"
    if not path.exists():
        return None
    saved = torch.load(path, map_location=device, weights_only=False)
    algo = saved.get("algorithm", "epsilon")
    net = NoisyDQN((4, 84, 84), 6) if algo == "noisynet" else DQN((4, 84, 84), 6)
    net.load_state_dict(saved["online"])
    return net.to(device).eval()


def load_pg(run_tag):
    path = P2_ROOT / run_tag / "best.pt"
    if not path.exists():
        return None
    saved = torch.load(path, map_location=device, weights_only=False)
    net = PGNet((4, 84, 84), 6, with_value=(saved["algorithm"] == "a2c"))
    net.load_state_dict(saved["model"])
    return net.to(device).eval()


def dqn_policy(net, epsilon, seed):
    rng = np.random.default_rng(seed)
    def fn(state, env):
        q = net(torch.from_numpy(state).unsqueeze(0).to(device)).squeeze(0)
        a = int(q.argmax().item()) if rng.random() >= epsilon else int(rng.integers(env.action_space.n))
        return a, float(q[a].item())
    return fn


def pg_policy(net, stochastic, seed):
    gen = torch.Generator(device=device).manual_seed(seed)
    def fn(state, env):
        logits, value = forward_fp32(net, torch.from_numpy(state).unsqueeze(0).to(device), False)
        if stochastic:
            a = int(torch.multinomial(torch.softmax(logits, -1), 1, generator=gen).item())
        else:
            a = int(logits.argmax(-1).item())
        return a, (float(value.item()) if value is not None else float("nan"))
    return fn


EVAL_PLAN = [
    # label,                       kind,  run_tag,                   mode
    ("DQN baseline (P1)",          "dqn", "baseline_epsilon_seed42", "greedy"),
    ("DQN baseline (P1)",          "dqn", "baseline_epsilon_seed42", "eps0.02"),
    ("DQN NoisyNet+PER (P1 best)", "dqn", "per_noisynet_seed42",     "greedy"),
    ("REINFORCE",                  "pg",  REINFORCE_TAG,             "greedy"),
    ("REINFORCE",                  "pg",  REINFORCE_TAG,             "stochastic"),
    ("A2C",                        "pg",  A2C_TAG,                   "greedy"),
    ("A2C",                        "pg",  A2C_TAG,                   "stochastic"),
]

EVAL_RESULTS = {}
for label, kind, tag, mode in EVAL_PLAN:
    cache = P2_ROOT / f"eval_{tag}_{mode}.json"
    if cache.exists() and not RERUN_EVAL:
        EVAL_RESULTS[(label, mode)] = json.loads(cache.read_text(encoding="utf-8"))
        print(f"{label:28s} {mode:10s} cached: {EVAL_RESULTS[(label, mode)]['mean']:+.2f}")
        continue
    net = load_dqn(tag) if kind == "dqn" else load_pg(tag)
    if net is None:
        print(f"{label:28s} {mode:10s} skipped (no best.pt for {tag})")
        continue
    if kind == "dqn":
        fn = dqn_policy(net, 0.02 if mode == "eps0.02" else 0.0, SEED)
    else:
        fn = pg_policy(net, mode == "stochastic", SEED)
    t0 = time.perf_counter()
    res = play_and_measure(fn)
    res.update({"label": label, "run_tag": tag, "mode": mode, "eval_minutes": (time.perf_counter() - t0) / 60})
    atomic_json(res, cache)
    EVAL_RESULTS[(label, mode)] = res
    print(f"{label:28s} {mode:10s} score {res['mean']:+.2f} ± {res['std']:.2f}  "
          f"bias {res.get('bias_mean', float('nan')):+.3f}")

## 16. Comparison of REINFORCE, A2C and DQN

All curves use the same definition: the mean reward of the last 100 training episodes, plotted
against agent steps, where 1 agent step = 4 emulator frames. Metrics:

| Metric | Meaning |
|---|---|
| frames / episodes / wall-min to +15 | convergence speed, using the same threshold as Project 1 |
| frames to solve | first time mean-100 ≥ 19 with ≥ 100 episodes (the Project 1 rule) |
| AUC@2M, AUC@10M | normalised area under the learning curve (Project 1 definition); higher means it learned sooner |
| gradient updates, samples in gradients | DQN reuses each transition many times from replay, while PG methods use each sample once |
| estimate bias | mean of (Q or V estimate − actual discounted return); > 0 means over-estimation |
| eval score | 30-episode evaluation of `best.pt` |

Wall-clock numbers depend on the hardware. The Project 1 runs used Colab L4/T4 GPUs with a single
environment, and the Project 2 runs use the local RTX 4080 with 16 environments, so the GPU column is
shown alongside them.

If the `CMPE260_Project1` folder is not present locally, the DQN rows are filled from the summary
numbers recorded in the Project 1 notebook outputs (`P1_SUMMARY` below). Those rows are marked
`(recorded)`, and metrics that were not recorded show "—".

In [ ]:
METHODS = {
    "DQN baseline (P1)":          dict(project=1, tag="baseline_epsilon_seed42",
                                       family="value-based, off-policy, 1-step TD + replay"),
    "DQN NoisyNet+PER (P1 best)": dict(project=1, tag="per_noisynet_seed42",
                                       family="value-based, off-policy, PER + parametric noise"),
    "REINFORCE":                  dict(project=2, tag=REINFORCE_TAG,
                                       family="policy gradient, on-policy, Monte-Carlo"),
    "A2C":                        dict(project=2, tag=A2C_TAG,
                                       family="actor-critic, on-policy, n-step TD"),
}


# Recorded in the saved outputs of CMPE260_DQN_Pong_Complete(PER).ipynb (Project 1).
P1_SUMMARY = {
    "baseline_epsilon_seed42": {
        "status": "max_frames", "frames": 2_000_000, "episodes": 1122,
        "final_reward_100": 18.97, "best_reward_100": 18.98, "wall_clock_minutes": 194.55,
        "gpu": "NVIDIA L4", "frames_to_15": 757_740, "frames_to_solve": None,
        "warmup_steps": 10_000, "batch_size": 32, "eval_other": "+19.73 ± 1.18 (ε=0.02)",
    },
    "per_noisynet_seed42": {
        "status": "solved", "frames": 376_858, "episodes": 215,
        "final_reward_100": 19.00, "best_reward_100": 19.00, "wall_clock_minutes": 61.75,
        "gpu": "Tesla T4", "frames_to_15": None, "frames_to_solve": 376_858,
        "warmup_steps": 10_000, "batch_size": 32, "eval_other": None,
    },
}


def p1_summary_row(label, meta):
    rec = P1_SUMMARY.get(meta["tag"])
    if rec is None:
        return None
    frames = rec["frames"]
    updates = max(frames - rec["warmup_steps"], 0)
    samples = updates * rec["batch_size"]
    return {
        "method": label + " (recorded)", "family": meta["family"], "status": rec["status"],
        "frames to +15": fmt(rec["frames_to_15"]), "episodes to +15": "—", "wall-min to +15": "—",
        "frames to solve (≥19)": fmt(rec["frames_to_solve"]),
        "best reward_100": f"{rec['best_reward_100']:+.2f}", "final reward_100": f"{rec['final_reward_100']:+.2f}",
        "AUC@2M": "—", "AUC@10M": "—",
        "total frames": fmt(frames), "emulator frames": fmt(4 * frames), "episodes": fmt(rec["episodes"]),
        "gradient updates": "≈" + fmt(updates), "samples in gradients": "≈" + fmt(samples),
        "sample reuse (samples/frame)": "≈" + fmt(samples / frames, "{:.2f}"),
        "wall-clock min": fmt(rec["wall_clock_minutes"], "{:,.1f}"),
        "agent steps / s": fmt(frames / (rec["wall_clock_minutes"] * 60), "{:,.0f}"),
        "GPU": rec["gpu"], "parameters": fmt(count_params(DQN((4, 84, 84), 6))),
        "eval greedy": "—", "eval ε/stochastic": rec["eval_other"] or "—",
        "value estimator": "Q(s,a)", "estimate bias (est − G)": "— (needs P1 best.pt)",
        "mean |est − G|": "—", "% steps over-estimated": "—",
    }


def p1_episode_log(run_tag, results):
    steps, rewards = tensorboard_curve(P1_ROOT / run_tag / "tensorboard", "episode/reward")
    if not len(steps):
        return None
    df = pd.DataFrame({"frame": steps, "reward": rewards})
    df["reward_100"] = df.reward.rolling(100, min_periods=1).mean()
    w_steps, w_vals = tensorboard_curve(P1_ROOT / run_tag / "tensorboard", "system/wall_minutes")
    if len(w_steps) > 1:
        df["wall_minutes"] = np.interp(df.frame, w_steps, w_vals)
        df.attrs["wall_estimated"] = False
    else:  # fall back to a constant-throughput estimate from results.json
        total = results.get("wall_clock_minutes", np.nan) if results else np.nan
        df["wall_minutes"] = df.frame / max(df.frame.iloc[-1], 1) * total
        df.attrs["wall_estimated"] = True
    return df


def first_crossing(df, threshold, min_episodes=1):
    ok = (df.reward_100.values >= threshold) & (np.arange(1, len(df) + 1) >= min_episodes)
    hits = np.flatnonzero(ok)
    if not hits.size:
        return None
    i = int(hits[0])
    return {"frame": int(df.frame.iloc[i]), "episodes": i + 1, "wall": float(df.wall_minutes.iloc[i])}


def fmt(x, spec="{:,.0f}"):
    return "—" if x is None or (isinstance(x, float) and not np.isfinite(x)) else spec.format(x)


rows, curves = [], {}
for label, meta in METHODS.items():
    root = P1_ROOT if meta["project"] == 1 else P2_ROOT
    rpath = root / meta["tag"] / "results.json"
    results = json.loads(rpath.read_text(encoding="utf-8")) if rpath.exists() else None
    log = p1_episode_log(meta["tag"], results) if meta["project"] == 1 else p2_episode_log(meta["tag"])
    if log is None or results is None:
        if meta["project"] == 1 and p1_summary_row(label, meta) is not None:
            rows.append(p1_summary_row(label, meta))
            print(f"{label}: Project 1 folder not found locally, using recorded summary numbers.")
        else:
            print(f"Skipping {label}: no logs/results yet.")
        continue
    curves[label] = log
    cfg = results.get("config", {})
    frames = int(results["frames"])
    if meta["project"] == 1:
        updates = max(frames - int(cfg.get("warmup_steps", 10_000)), 0)
        samples = updates * int(cfg.get("batch_size", 32))
        approx = "≈"
        n_params = count_params(DQN((4, 84, 84), 6))
    else:
        updates, samples, approx = int(results["updates"]), int(results["samples_used"]), ""
        n_params = int(results["num_params"])
    c15 = first_crossing(log, 15.0)
    c19 = first_crossing(log, 19.0, min_episodes=100)
    wall_min = float(results["wall_clock_minutes"])
    greedy = EVAL_RESULTS.get((label, "greedy"))
    other = EVAL_RESULTS.get((label, "stochastic")) or EVAL_RESULTS.get((label, "eps0.02"))
    bias_src = EVAL_RESULTS.get((label, "stochastic")) if "A2C" in label else greedy
    rows.append({
        "method": label,
        "family": meta["family"],
        "status": results["status"],
        "frames to +15": fmt(c15 and c15["frame"]),
        "episodes to +15": fmt(c15 and c15["episodes"]),
        "wall-min to +15": fmt(c15 and c15["wall"], "{:,.1f}") + ("*" if log.attrs.get("wall_estimated") else ""),
        "frames to solve (≥19)": fmt(c19 and c19["frame"]),
        "best reward_100": fmt(float(log.reward_100.max()), "{:+.2f}"),
        "final reward_100": fmt(float(log.reward_100.iloc[-1]), "{:+.2f}"),
        "AUC@2M": fmt(normalized_auc(log.frame, log.reward_100, 2_000_000), "{:.3f}"),
        "AUC@10M": fmt(normalized_auc(log.frame, log.reward_100, 10_000_000), "{:.3f}"),
        "total frames": fmt(frames),
        "emulator frames": fmt(4 * frames),
        "episodes": fmt(len(log)),
        "gradient updates": approx + fmt(updates),
        "samples in gradients": approx + fmt(samples),
        "sample reuse (samples/frame)": approx + fmt(samples / max(frames, 1), "{:.2f}"),
        "wall-clock min": fmt(wall_min, "{:,.1f}"),
        "agent steps / s": fmt(frames / max(wall_min * 60, 1e-9), "{:,.0f}"),
        "GPU": results.get("gpu", "?"),
        "parameters": fmt(n_params),
        "eval greedy": "—" if greedy is None else f"{greedy['mean']:+.2f} ± {greedy['std']:.2f}",
        "eval ε/stochastic": "—" if other is None else f"{other['mean']:+.2f} ± {other['std']:.2f}",
        "value estimator": {"REINFORCE": "none (MC return)", "A2C": "V(s) critic"}.get(label, "Q(s,a)"),
        "estimate bias (est − G)": "n/a (unbiased MC)" if label == "REINFORCE" else (
            "—" if not bias_src or "bias_mean" not in bias_src else f"{bias_src['bias_mean']:+.3f}"),
        "mean |est − G|": "—" if not bias_src or "abs_error_mean" not in bias_src else f"{bias_src['abs_error_mean']:.3f}",
        "% steps over-estimated": "—" if not bias_src or "overestimated_fraction" not in bias_src
                                   else f"{100 * bias_src['overestimated_fraction']:.0f}%",
    })

COMPARISON = pd.DataFrame(rows).set_index("method")
COMPARISON.to_csv(P2_ROOT / "comparison_table.csv", encoding="utf-8-sig")  # Excel-friendly UTF-8
pd.set_option("display.max_colwidth", 60)
display(COMPARISON.T)
try:
    (P2_ROOT / "comparison_table.md").write_text(COMPARISON.T.to_markdown(), encoding="utf-8")
except ImportError:
    pass
print("* wall-clock estimated from total run time (no wall_minutes tag in that log).")
print("Saved:", P2_ROOT / "comparison_table.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for label, log in curves.items():
    axes[0].plot(log.frame / 1e6, log.reward_100, label=label, lw=1.6)
    axes[1].plot(log.wall_minutes, log.reward_100, label=label, lw=1.6)
for ax, xl in zip(axes, ["agent steps (millions)", "wall-clock minutes"]):
    ax.axhline(15, ls=":", c="gray", lw=1)
    ax.axhline(19, ls="--", c="gray", lw=1)
    ax.set_xlabel(xl)
    ax.set_ylabel("mean reward (last 100 episodes)")
    ax.set_ylim(-21.5, 21.5)
    ax.grid(alpha=0.3)
axes[0].set_title("Sample efficiency")
axes[1].set_title("Wall-clock efficiency")
axes[0].legend(loc="lower right")
plt.tight_layout()
plt.savefig(P2_ROOT / "learning_curves.png", dpi=150)
plt.show()

# Value-estimate vs realised discounted return (bias / over-estimation)
bias_sets = [(k, v) for k, v in EVAL_RESULTS.items()
             if "sample_estimates" in v and ((k[0].startswith("DQN") and k[1] == "greedy")
                                             or (k[0] == "A2C" and k[1] == "stochastic"))]
if bias_sets:
    fig, axes = plt.subplots(1, len(bias_sets), figsize=(5 * len(bias_sets), 4.5), squeeze=False)
    for ax, ((label, mode), res) in zip(axes[0], bias_sets):
        ax.scatter(res["sample_returns"], res["sample_estimates"], s=3, alpha=0.3)
        lim = [-1.05, 1.05]
        ax.plot(lim, lim, "k--", lw=1)
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xlabel("actual discounted return G_t")
        ax.set_ylabel("estimate (Q or V)")
        ax.set_title(f"{label} [{mode}]\nbias={res['bias_mean']:+.3f}, corr={res['corr']:.2f}")
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(P2_ROOT / "value_bias.png", dpi=150)
    plt.show()

for prefix in ("reinforce", "a2c"):
    path = P2_ROOT / f"sweep_{prefix}.csv"
    if path.exists():
        print(f"\nHyper-parameter sweep: {prefix}")
        display(pd.read_csv(path))

## 17. Interpreting the results for the report

Fill in the discussion from the measured table above, not from expectations. The points below are
what to look for and why each one happens.

* **Speed of convergence (frames).** DQN reuses every transition about 32 times from replay
  (`sample reuse`), so it usually needs the fewest environment frames. A2C uses each sample once,
  but its bootstrapped, low-variance advantages let it learn steadily. REINFORCE waits for complete
  returns and has the noisiest gradient, so it usually needs the most frames. Compare
  `frames to +15` and `AUC@2M`.
* **Speed of convergence (wall-clock).** The PG methods step 16 environments at once and make cheap
  updates, so they run at thousands of agent steps per second, against about 100–170 for Project 1's
  single-environment DQN that updates every step. A method can therefore need more frames and still
  finish sooner. Compare `wall-min to +15` and `agent steps / s`, and note the GPU column.
* **Bias / over-estimation.** DQN's `max` target over-estimates value: expect a positive
  `estimate bias` and a training-time `q_mean` above the realised returns. A2C's critic is
  bootstrapped, so it is biased while it is inaccurate, but it has no max operator. REINFORCE is
  unbiased and pays for that in variance (`train/return_std`, `train/grad_norm`).
* **Samples and updates.** DQN makes about 2M small updates (32 samples each). A2C makes about 10⁵
  updates of 80 samples. REINFORCE makes about 10³ updates of up to 8k samples.
* **Stability.** Look at how smooth each curve is and the eval standard deviation. On-policy PG
  curves are smoother per update but sensitive to the learning rate, as the sweep tables show.

All runs use one seed (42), as in Project 1, so the differences are suggestive rather than
statistically conclusive. Say this in the report.

### References
1. Williams (1992), *Simple statistical gradient-following algorithms for connectionist RL* (REINFORCE).
2. Mnih et al. (2016), *Asynchronous Methods for Deep Reinforcement Learning* (A3C/A2C).
3. Schulman et al. (2016), *High-Dimensional Continuous Control Using GAE*.
4. Mnih et al. (2015), *Human-level control through deep RL* (DQN).
5. Karpathy (2016), *Deep Reinforcement Learning: Pong from Pixels*.